In [22]:
import pandas as pd
import os
import sys

from ucimlrepo import fetch_ucirepo 
import sklearn

from chronoepilogi import ChronoEpilogi
from models import LearningModel

# Autoregressive forecasting

Given the past energy consumption and other features (up to time t), predict time t+1 of the energy consumption.

In [23]:
# fetch dataset 
appliances_energy_prediction = fetch_ucirepo(id=374) 
# data (as pandas dataframes) 
X = appliances_energy_prediction.data.features 
y = appliances_energy_prediction.data.targets
# merge
X = pd.merge(X,y,left_index=True, right_index=True)
# remove date
X = X[X.columns[1:]]

In [24]:
numerical_columns = X.columns
categorical_columns = []
variable_types = {**dict([(c,"numerical") for c in numerical_columns]),**dict([(c,"categorical") for c in categorical_columns])}
target_type = "continuous"
target = "Appliances"

Forecast one step ahead with one hour of data (6 lags)

In [25]:
fs_instance = ChronoEpilogi(X, target,
                            phases="FBEV",
                            target_type=target_type,
                            default_max_lag=6,
                            variable_types=variable_types)
fs_instance.fit()
print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)

selected set: ['Appliances', 'lights', 'RH_out', 'RH_5', 'RH_1', 'Windspeed', 'T1', 'T3']
length of selected set: 8 and number of covariates in the dataset: 27
number of Markov Boundaries: 4
{'Appliances': ['Appliances'], 'lights': ['lights'], 'RH_out': [np.str_('RH_2'), np.str_('T6'), np.str_('T2'), 'RH_out'], 'RH_5': ['RH_5'], 'RH_1': ['RH_1'], 'Windspeed': ['Windspeed'], 'T1': ['T1'], 'T3': ['T3']}


Less concervative equivalence test by disabling the Early Stopping of equivalences:

In [26]:
fs_instance = ChronoEpilogi(X, target,
                            phases="FBEV",
                            equivalence_early_stopping=False,
                            target_type=target_type,
                            default_max_lag=6,
                            variable_types=variable_types)
fs_instance.fit()
print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())
print(fs_instance.equivalent_variables)

selected set: ['Appliances', 'lights', 'RH_out', 'RH_5', 'RH_1', 'Windspeed', 'T1', 'T3']
length of selected set: 8 and number of covariates in the dataset: 27
number of Markov Boundaries: 4
{'Appliances': ['Appliances'], 'lights': ['lights'], 'RH_out': ['T2', 'RH_2', 'T6', 'RH_out'], 'RH_5': ['RH_5'], 'RH_1': ['RH_1'], 'Windspeed': ['Windspeed'], 'T1': ['T1'], 'T3': ['T3']}


# Non-autoregressive forecasting

Setting in which (a lookback window of) the past of covariates is provided, but not the past of the forecasted quantity.

In the original setting, only information at time t was used to predict occupancy at time t.
Since ChronoEpilogi current implementation does not handle contemporaneous data, y is shifted by 1 into the future.

In [27]:
# fetch dataset 
room_occupancy_estimation = fetch_ucirepo(id=864) 
# data (as pandas dataframes) 
X = room_occupancy_estimation.data.features 
y = room_occupancy_estimation.data.targets 
# removing date and time
X = X[X.columns[2:]]

# realign data so that target is at time t+1
X.index = list(range(len(X)))
y.index = [i+1 for i in X.index]

# merge
X = pd.merge(X,y,left_index=True, right_index=True)

In [28]:
numerical_columns = X.columns[:14]
categorical_columns = X.columns[14:]
variable_types = {**dict([(c,"numerical") for c in numerical_columns]),**dict([(c,"categorical") for c in categorical_columns])}
target_type = "count"
target = "Room_Occupancy_Count"
start_with_univariate_autoregressive_model = False  # provides the information that the model should not be autoregressive. Perhaps I should change that name.

ChronoEpilogi with only lag 1, corresponding to the original setting

In [29]:
fs_instance = ChronoEpilogi(X, target,
                            phases="FBEV",
                            start_with_univariate_autoregressive_model=start_with_univariate_autoregressive_model,
                            target_type=target_type,
                            default_k=1,
                            default_max_lag=1,
                            variable_types=variable_types)
fs_instance.fit()
print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())

selected set: ['S1_Temp', 'S1_Light', 'S7_PIR', 'S5_CO2_Slope', 'S6_PIR', 'S3_Sound', 'S2_Light', 'S4_Light', 'S4_Sound', 'S2_Sound', 'S4_Temp', 'S3_Light', 'S1_Sound', 'S3_Temp', 'S5_CO2', 'S2_Temp']
length of selected set: 16 and number of covariates in the dataset: 16
number of Markov Boundaries: 1


ChronoEpilogi with the past 10 minutes (20 lags)

In [30]:
fs_instance = ChronoEpilogi(X, target,
                            phases="FBEV",
                            start_with_univariate_autoregressive_model=start_with_univariate_autoregressive_model,
                            target_type=target_type,
                            default_k=4,
                            default_max_lag=20,
                            variable_types=variable_types)
fs_instance.fit()
print("selected set:", fs_instance.selected_set)
print("length of selected set:",len(fs_instance.selected_set),"and number of covariates in the dataset:",len(X.columns) -1)
print("number of Markov Boundaries:",fs_instance.get_total_number_sets())

selected set: ['S1_Temp', 'S5_CO2_Slope', 'S6_PIR', 'S4_Sound', 'S1_Light', 'S2_Light', 'S7_PIR', 'S2_Sound', 'S2_Temp', 'S1_Sound', 'S3_Temp', 'S3_Light']
length of selected set: 12 and number of covariates in the dataset: 16
number of Markov Boundaries: 1


# Tabular data (with no temporal index)

In [31]:
from ucimlrepo import fetch_ucirepo 
  
# fetch dataset 
ai4i_2020_predictive_maintenance_dataset = fetch_ucirepo(id=601) 
# base tabular data
X = ai4i_2020_predictive_maintenance_dataset.data.features 
y = ai4i_2020_predictive_maintenance_dataset.data.targets 
# select overall machine failure
y = y[["Machine failure"]]

# multiindex data considers the predicted quantity to be at the same index as the predictors
# so no need to realign data
X.index = list(range(len(X)))
y.index = X.index

# put X into multilevel column index format
X.columns = pd.MultiIndex.from_tuples([(c,"-1") for c in X.columns])
y.columns = pd.MultiIndex.from_tuples([(c,None) for c in y.columns])
# merge data
X = pd.merge(X,y,left_index=True, right_index=True)

# float for all numerical types
X["Rotational speed"] = X["Rotational speed"].astype(float)
X["Tool wear"] = X["Tool wear"].astype(float)
# mapping to 0,1,2 for type
X[("Type","-1")] = X[("Type","-1")].apply(lambda x: {"L":0,"M":1,"H":2}[x])
  

In [32]:
variable_types = {"Type":"categorical","Air temperature":"numerical","Process temperature":"numerical",
                  "Rotational speed":"numerical","Torque":"numerical","Tool wear":"numerical"}
target_type = "binary"
target = ("Machine failure",None)

In [33]:
fs_instance = ChronoEpilogi(X, target,
                            target_type=target_type,
                            variable_types=variable_types)
fs_instance.fit()
fs_instance.selected_set

['Torque',
 'Tool wear',
 'Rotational speed',
 'Air temperature',
 'Process temperature',
 'Type']

In [34]:
fs_instance = ChronoEpilogi(X, target,
                            phases="FBEV",
                            target_type=target_type,
                            variable_types=variable_types)
fs_instance.fit()
fs_instance.equivalent_variables

{'Torque': ['Torque'],
 'Tool wear': ['Tool wear'],
 'Rotational speed': ['Rotational speed'],
 'Air temperature': ['Air temperature'],
 'Process temperature': ['Process temperature'],
 'Type': ['Type']}

# Custom model

Let's assume that I want to analyse the previous dataset and that I want:
 1) to use a double-level column index dataframe
 2) to use the linear regression model from sklearn
 3) to obtain exactly 3 covariates, and not stop earlier or later
 4) to obtain a single solution without computing equivalences

I may create the following model:

In [35]:
class MyCustomModel(LearningModel):
    def _prepare_data(self, data):
        X = data[[column for column in data.columns if column!=self.target]].to_numpy()
        y = data[self.target].to_numpy()
        return X,y

    def fit(self, data):
        self.data = data
        X,y = self._prepare_data(data)
        self.model = sklearn.linear_model.LinearRegression().fit(X,y)

    def fittedvalues(self, data=None):
        if data is None:
            if self.data is None:
                raise(RuntimeError)
            data = self.data
        
        X,_ = self._prepare_data(data)
        y_pred = self.model.predict(X)
        return pd.Series(y_pred, index = data.index)
        
    def stopping_metric(self, previous_model, method=""):
        # we care only for the number of TS, so it is fine to leave this empty
        return 0.
    def has_too_many_parameters(self, ratio):
        # we care only for the number of TS, so it is fine to leave this empty
        # if too few observations are given sklearn will throw an error anyways
        return False

To apply it, I need to specify the model_class and model_config.
Here, model_config is empty as I did not specify a parameter configuration for my model.

To set the maximal number of covariates to reach at 3, I set the parameter maximal_selected_size.

To avoid computing equivalences, I set the algorithm to only operate a forward-backward phase with phases = "FB".

In [36]:
fs_instance = ChronoEpilogi(X, target,
                            phases="FB",
                            target_type=target_type,
                            variable_types=variable_types,
                            model_class = MyCustomModel,
                            model_config = dict(),
                            maximal_selected_size=3)
fs_instance.fit()
fs_instance.selected_set

['Torque', 'Rotational speed', 'Tool wear']